<a href="https://vigneashpandiyan.github.io/publications/Codes/" target="_blank" rel="noopener noreferrer">
  <img src="https://vigneashpandiyan.github.io/images/Link.png"
       style="max-width: 800px; width: 100%; height: auto;">
</a>

Datasets and Dataloaders in pytorch
===================================

Data sets can be thought of as big arrays of data.  If the data set is small enough (e.g., MNIST, which has 60,000 28x28 grayscale images), a dataset can be literally represented as an array - or more precisely, as a single pytorch tensor.  With one number per pixel, MNIST takes about 200 megabytes of RAM, which fits comfortably into a modern computer.

But larger-scale datasets like ImageNet or Places365 have more than a million higher-resolution full-color images.  In these cases, an ordinary python array or pytorch tensor would require more than a terabyte of RAM, which is impractical on most computers.

Instead, we need to load the data from disk (or SSD).  Unfortunately, the latency of loading from disk is very slow compared to RAM, so we need to do the loading cleverly if we want to load the data quickly.

To solve the problem, pytorch provides two classes:
 * `torch.utils.data.Dataset` - This very simple base class represents an array where the actual data may be slow to fetch, typically because the data is in disk files that require some loading, decoding, or other preprocessing. Pytorch provides a variety of different `Dataset` subclasses.  For example, there is a handy one called `ImageFolder` that treats a directory tree of image files as an array of classified images.
 * `torch.utils.data.DataLoader` - This fancy class wraps a `Dataset` as a stream of data batches.  Behind the scenes it uses a few techniques to feed the data faster.  You do not need to subclass `DataLoader` - its purpose is to make a `Dataset` speedy.

<img src="https://github.com/davidbau/how-to-read-pytorch/blob/master/notebooks/dataloader.png?raw=1" style="max-width:100%">

In [ ]:
# To save time, start this download first, before reading through the examples.
import torch, torchvision, os
if not os.path.isfile('datasets/miniplaces/train/yard/00001000.jpg'):
    torchvision.datasets.utils.download_and_extract_archive(
        'http://dissect.csail.mit.edu/datasets/miniplaces.zip',
        'datasets', md5='bfabeb497c7eca01c74cd8441a9ac108')

## Looking at an image data set using ImageFolder

The most common `Dataset` used in computer vision is `ImageFolder`, which loads a set of image files from a directory tree.  It treats every subdirectory of images as a classification category.  To demonstrate it, we will use it to load images from the miniplaces dataset loaded above.

**Directory layout.** Notice that `datasets/miniplaces/val` contains a set of 100 directories with names like `golf_course`.  Each of these directories contains 100 images, each stored as a jpeg file: 10000 images in total.

In [ ]:
ls datasets/miniplaces/val/golf_course

**Constructing an ImageFolder.**  Making an ImageFolder at the root directory of the dataset creates an object that behaves like an array: it has a length, and each entry contains a tuple with an image and a number.  The image is stored as a `PIL` object which is a standard python object for images, and the number denotes the classification class - with one class for each folder, numbered in alphabetical order.

In [ ]:
val_set = torchvision.datasets.ImageFolder('datasets/miniplaces/val')
print('Length is', len(val_set))
item = val_set[5100]
print('5100th item is a pair', item)

# Display the PIL image and the class name directly.
display(item[0])
print('Class name is', val_set.classes[item[1]])

**Transforming the PIL image into a pytorch tensor.**  A PIL image is not convenient for training: we would prefer our data set to return pytorch tensors.  So we can tell `ImageFolder` to do this by specifying the `transform` function on construction.  Pytorch comes with a standard transform function `torchvision.transforms.ToTensor()` which converts an image to a pytorch tensor.

Now when indexing into the data set, we will get a pytorch tensor instead of a PIL image.

In [ ]:
import matplotlib.pyplot as plt

# Define the ToTensor transform
to_tensor_transform = torchvision.transforms.ToTensor()

# Create an ImageFolder dataset without transform to get the original image
val_set_untransformed = torchvision.datasets.ImageFolder('datasets/miniplaces/val')

# Create an ImageFolder dataset with ToTensor transform
val_set_transformed = torchvision.datasets.ImageFolder('datasets/miniplaces/val',
                                                      transform=to_tensor_transform)

index_to_show = 5100 # Using the same index as in UeC9gGdTsoso for consistency

# --- Display original image (before transformation) ---
original_image, original_label = val_set_untransformed[index_to_show]
print(f"Image at index {index_to_show} BEFORE ToTensor transformation:")
display(original_image)
print(f"Class name: {val_set_untransformed.classes[original_label]}")
# Removed: print(f"Tensor shape: {original_image.shape}") as PIL images do not have a .shape attribute
print("--------------------------------------------------")

# --- Display transformed image (after ToTensor, converted back to PIL for visualization) ---
transformed_tensor, transformed_label = val_set_transformed[index_to_show]
print(f"Image at index {index_to_show} AFTER ToTensor transformation:")
print(f"Tensor shape: {transformed_tensor.shape}")

# Inverse transform to display the tensor as a PIL image
as_image = torchvision.transforms.ToPILImage()
display(as_image(transformed_tensor))
print(f"Class name: {val_set_transformed.classes[transformed_label]}")

## Improving Training using Data Augmentation

One of the main ways to stretch a dataset to make it more effective for training is to randomly adjust the images.  For example if we randomly adjust the crop, color, or orientation of the image while loading, using the same image file multiple times will produce different training examples for the network.  This is an easy way to increase the amount of training diversity in the data set without requring more actual images.

To do data augmentation in a pytorch `Dataset`, you can specify more operations on `transform=` besides `ToTensor()`.

In particular, there is a `Compose` transform that makes it easy to chain a series of data transformations; and `torchvision.transforms` includes a number of useful image transforms such as random resized crops and image flips.

Here is an example (you may run it multiple times to see more variations):

### Demonstrating RandomCrop and RandomHorizontalFlip


In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

# Define the transformations
augmentation_transforms = transforms.Compose([
    transforms.RandomCrop(112),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), # Add ToTensor to convert to tensor for consistency
])

# Reuse the untransformed validation set from previous examples (cell kOTNP5BFs26e)
# If val_set_untransformed is not defined, you might need to re-run cell kOTNP5BFs26e first
try:
    if 'val_set_untransformed' not in locals():
        val_set_untransformed = torchvision.datasets.ImageFolder('datasets/miniplaces/val')
except NameError:
    val_set_untransformed = torchvision.datasets.ImageFolder('datasets/miniplaces/val')

# Get an original image (PIL Image) to demonstrate the transformations
original_image, original_label = val_set_untransformed[5100] # Using index 5100 as before

print(f"Original Image (PIL) from class: {val_set_untransformed.classes[original_label]}")
display(original_image)

# Display several augmented versions of the image
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Image after RandomCrop and RandomHorizontalFlip (4 variations)', fontsize=16)

for i in range(4):
    transformed_tensor = augmentation_transforms(original_image)
    # Convert back to PIL Image for display
    display_image = transforms.ToPILImage()(transformed_tensor)
    axes[i].imshow(display_image)
    axes[i].set_title(f'Variation {i+1}')
    axes[i].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

### Demonstrating Color to Black and White (Grayscale) Transformation

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torchvision

# Ensure we have an untransformed dataset to get the original PIL image
try:
    if 'val_set_untransformed' not in locals():
        val_set_untransformed = torchvision.datasets.ImageFolder('datasets/miniplaces/val')
except NameError:
    val_set_untransformed = torchvision.datasets.ImageFolder('datasets/miniplaces/val')

# Get an original image (PIL Image) to demonstrate the transformation
original_image, original_label = val_set_untransformed[5100] # Using index 5100 as before

# Define the Grayscale transformation
grayscale_transform = transforms.Grayscale(num_output_channels=1)

# Apply the transformation
grayscale_image = grayscale_transform(original_image)

# Display the original and grayscale images
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('Image Before and After Grayscale Conversion', fontsize=16)

axes[0].imshow(original_image)
axes[0].set_title('Original Color Image')
axes[0].axis('off')

axes[1].imshow(grayscale_image, cmap='gray') # Use 'gray' colormap for grayscale images
axes[1].set_title('Grayscale Image')
axes[1].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

Fast Dataset Access using DataLoader

When we use a dataset for training, we will usually run through the whole dataset in batches. We could do this ourselves by just fetching the images one at a time and grouping them.

But a faster way to iterate through the dataset is to wrap our val_set object in a torch.utils.data.DataLoader object. The val_loader we get can magically pull data out of the Dataset much faster than doing it in the simple way; the DataLoader class does this by using several threads to load and prefetch the data.

The speedup will depend on the system and the number of threads you use (the number of threads to use is specified using num_workers). In practice using DataLoader will typically be 5-20 times faster than direct Dataset access.

In [ ]:
import time
import torch
import torchvision.transforms as transforms

# Ensure val_set is initialized with ToTensor transform
# This assumes val_set_transformed is available from previous cells or re-create it
# If val_set_transformed is not defined, run cell kOTNP5BFs26e first.

try:
    # Use the val_set that has ToTensor applied from cell kOTNP5BFs26e
    if 'val_set_transformed' not in locals():
        to_tensor_transform = transforms.ToTensor()
        val_set_for_timing = torchvision.datasets.ImageFolder('datasets/miniplaces/val',
                                                            transform=to_tensor_transform)
    else:
        val_set_for_timing = val_set_transformed
except NameError:
    # Fallback if val_set_transformed was not created
    to_tensor_transform = transforms.ToTensor()
    val_set_for_timing = torchvision.datasets.ImageFolder('datasets/miniplaces/val',
                                                        transform=to_tensor_transform)


print('Going over the data set as an array.')
start = time.time()
summed_image_dataset = 0
batch_size = 100
for i in range(0, len(val_set_for_timing), batch_size):
    # Now val_set_for_timing[i+j][0] will return a tensor
    image_batch = torch.stack([val_set_for_timing[i+j][0] for j in range(batch_size)])
    summed_image_dataset += image_batch.sum(0)
end = time.time()
print(f'Took {end - start} seconds')

print('Going over the same dataset using a dataloader.')
start = time.time()
# Use val_set_for_timing for consistency
val_loader = torch.utils.data.DataLoader(
    val_set_for_timing, batch_size=batch_size, num_workers=2)
summed_image_loader = 0
for image_batch, label_batch in val_loader:
    summed_image_loader += image_batch.sum(0)
end = time.time()
print(f'Took {end - start} seconds')

# RoadMap  - Custom Data Loader Examples

Example dataloaders for various dataset storage types:

    1. Multi Class Image Classifier - Foldered Dataset
    2. Multi Class Image Classifier - Load labels and path from CSV
    3. Multi Class Multi Label Image Classifier

In [ ]:
# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from torchvision import transforms, datasets
from PIL import Image
import cv2
%matplotlib inline
from matplotlib import pyplot as plt



import os
import pandas as pd
from skimage import io, transform
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils


plt.ion()   # interactive mode

## 1. Multi Class Image Classifier - Foldered Dataset
    Data storage directory
        [NOTE: Directory and File names can be anything]
        Parent Directory
            |
            |----Train
            |       |
            |       |----Class1
            |       |       |----img1.png
            |       |       |----img2.png
            |       |----Class2
            |       |       |----img1.png
            |               |----img2.png
            |-----Val
            |       |
            |       |----Class1
            |       |       |----img1.png
            |       |       |----img2.png
            |       |----Class2
            |       |       |----img1.png
            |       |       |----img2.png

In [ ]:
import os
import requests
import zipfile
from torchvision import datasets, transforms

# 1. Setup paths
url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
zip_filename = "hymenoptera_data.zip"
extract_folder = "." # Extracts to current directory

# 2. Download the file
if not os.path.exists(zip_filename):
    print("Downloading dataset...")
    response = requests.get(url, stream=True)
    with open(zip_filename, 'wb') as file:
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                file.write(chunk)
    print("Download complete.")
else:
    print("Zip file already exists.")

# 3. Extract the zip file
# This usually creates a folder named 'hymenoptera_data' containing 'train' and 'val'
if not os.path.exists("hymenoptera_data"):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

# 4. Define Transforms
# ImageFolder requires a transform to convert images to tensors
data_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 5. Load the Data
# Note: The zip extracts to 'hymenoptera_data/train', not 'sample_data/train'
train_dir = os.path.join('hymenoptera_data', 'train')

train_data = datasets.ImageFolder(root=train_dir, transform=data_transform)

print(f"Successfully loaded {len(train_data)} training images.")

In [ ]:
# Getting class list
class_list =  train_data.classes
print(class_list)

In [ ]:
# Image class names to IDs
class_to_id = train_data.class_to_idx
print(class_to_id)

In [ ]:
# Getting images list with associated class ids
image_list = train_data.imgs
print(image_list)

In [ ]:
# Getting list of transforms applied
transforms_list = train_data.transform
print(transforms_list)

In [ ]:
for image, label in train_data:
    print(image.size(), label);
    # Transpose the image tensor from (C, H, W) to (H, W, C)
    # .permute(1, 2, 0) rearranges the dimensions.
    # .numpy() converts the PyTorch tensor to a NumPy array for matplotlib.
    plt.imshow(image.permute(1, 2, 0).numpy())

    break;

In [ ]:
#transformations
#uncommenting ToTensor()
data_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

In [ ]:
#Using Image Folder Function
train_data = datasets.ImageFolder(root=train_dir, transform=data_transform)

In [ ]:
#Using DataLoader
train_loader = torch.utils.data.DataLoader(train_data,
                                             batch_size=2, shuffle=True,
                                             num_workers=4)

In [ ]:
#batch size = 2
for data, labels in train_loader:
    print(data.shape, labels.shape);

    break;

## 2. Multi Class Image Classifier - Load labels and path from CSV
     
    Data storage directory
        [NOTE: Directory and File names can be anything]
        Parent Directory
            |
            |----Train
            |       |
            |       |----Images
            |       |       |----img1.png
            |       |       |----img2.png
            |       |----train_labels.csv
            |
            |----Val
            |       |
            |       |----Images
            |       |       |----img1.png
            |       |       |----img2.png
            |       |----val_labels.csv
            
            
    train_labels.csv contains a header row, the headers could be anything - one for image name column and another for the class id
    subsequent rows will be each filled with image names and labels
   

In [ ]:
import os
import requests
import pandas as pd
import random

# --- CONFIG ---
root_dir = "my_multiclass_dataset"
img_dir = os.path.join(root_dir, "images")
csv_path = os.path.join(root_dir, "train_labels.csv")

os.makedirs(img_dir, exist_ok=True)

# Classes to simulate
classes = ["cat", "dog", "bird"]
# Placeholder image URLs (using fixed IDs for stability)
urls = {
    "cat": "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=200",
    "dog": "https://images.unsplash.com/photo-1517849845537-4d257902454a?w=200",
    "bird": "https://images.unsplash.com/photo-1444464666168-49d633b86797?w=200"
}

data = []

print(f"Downloading dataset to '{root_dir}'...")

for i in range(50):
    # Assign a random class
    label = classes[i % 3]
    filename = f"image_{i:03d}.jpg"
    filepath = os.path.join(img_dir, filename)

    # Download image
    if not os.path.exists(filepath):
        try:
            r = requests.get(urls[label], timeout=10)
            with open(filepath, "wb") as f:
                f.write(r.content)
        except Exception as e:
            print(f"Error downloading {filename}: {e}")
            continue

    # Add to CSV record
    data.append([filename, label])

# Create CSV
df = pd.DataFrame(data, columns=["filename", "label"])
df.to_csv(csv_path, index=False)

print("Done! Dataset created.")
print(df.head())

In [ ]:
train_images_folder = "my_multiclass_dataset/images";
train_csv = "my_multiclass_dataset/train_labels.csv";

In [ ]:
# Create a custom Image Dataset Class

#img_list = list of images
#label_list = list of labels in the same order
#prefix = relative path to images folder

class DatasetMultiClassCSV(Dataset):
    def __init__(self, img_list, label_list, prefix, transform=None):
        self.img_list = img_list;
        self.label_list = label_list;
        self.transform = transform;
        self.prefix = prefix;
        self.classes = sorted(np.unique(label_list));
        self.class_to_idx = {};
        for i in range(len(self.classes)):
            self.class_to_idx[self.classes[i]] = i;

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, index):
        image_name = self.prefix + "/" + self.img_list[index];
        image = Image.open(image_name).convert('RGB');
        label = int(self.classes.index(self.label_list[index]));
        if self.transform is not None:
            image = self.transform(image);
        return image, label

In [ ]:
# Read the csv

import pandas as pd
df = pd.read_csv(train_csv);

img_list = [];
label_list = [];

for i in range(len(df)):
    img_list.append(df["filename"][i]);
    label_list.append(df["label"][i]);

In [ ]:
#transformations

#setting ToTensor() commented for better visualization
data_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        #transforms.ToTensor(),
        #transforms.Normalize(mean=[0.485, 0.456, 0.406],
        #                     std=[0.229, 0.224, 0.225])
    ])

#Using Image Folder Function
train_data = DatasetMultiClassCSV(img_list, label_list, train_images_folder,
                                           transform=data_transform)

In [ ]:
# Getting class list
class_list =  train_data.classes
print(class_list)

In [ ]:
# Image class names to IDs
class_to_id = train_data.class_to_idx
print(class_to_id)

In [ ]:
# Getting list of transforms applied
transforms_list = train_data.transform
print(transforms_list)

In [ ]:
for image, label in train_data:
    print(image.size, label);
    plt.imshow(image)

    break;

In [ ]:
#transformations
#uncommenting ToTensor()
data_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

In [ ]:
#Using Image Folder Function
train_data = DatasetMultiClassCSV(img_list, label_list, train_images_folder,
                                           transform=data_transform)

In [ ]:
#Using DataLoader
train_loader = torch.utils.data.DataLoader(train_data,
                                             batch_size=2, shuffle=True,
                                             num_workers=4)

In [ ]:
#batch size = 2
for data, labels in train_loader:
    print(data.shape, labels.shape);

    break;

## 3. Multi Class Multi Label Image Classifier  - One hot encoding

    Data storage directory
        [NOTE: Directory and File names can be anything]
        Parent Directory
            |
            |----Train
            |       |
            |       |----Images
            |       |       |----img1.png
            |       |       |----img2.png
            |       |----train_labels.csv
            |
            |----Val
            |       |
            |       |----Images
            |       |       |----img1.png
            |       |       |----img2.png
            |       |----val_labels.csv
            
            
    train_labels.csv contains a header row, the headers could be anything - one for image name column and another for the list of classes it represents
    subsequent rows will be each filled with image names and list of labels
    
    # A sample of data is taken from https://analyticsindiamag.com/multi-label-image-classification-with-tensorflow-keras/ for demonstration


In [ ]:
import os
import requests
import pandas as pd
import random

# --- CONFIGURATION ---
dataset_root = "multi_label_dataset"
images_dir = os.path.join(dataset_root, "Images")
csv_path = os.path.join(dataset_root, "train_labels.csv")

# Create folders
os.makedirs(images_dir, exist_ok=True)

# List of classes (Multi-label definitions)
# We will download images representing these categories
categories = ["Desert", "Forest", "Mountain"]
urls = {
    "Desert": "https://images.unsplash.com/photo-1509316975850-ff9c5deb0cd9?w=300",
    "Forest": "https://images.unsplash.com/photo-1448375240586-dfd8d395ea6c?w=300",
    "Mountain": "https://images.unsplash.com/photo-1464822759023-fed622ff2c3b?w=300"
}

data_records = []

print(f"Starting automatic download to: {dataset_root}...")

# --- DOWNLOAD & LABEL GENERATION ---
# We will generate 15 images (5 of each primary type)
# But strictly treat them as multi-label (One-Hot Encoded)
total_images = 15

for i in range(total_images):
    # Select a primary category for the image content
    primary_cat = categories[i % 3]
    filename = f"image_{i:03d}.jpg"
    file_path = os.path.join(images_dir, filename)

    # 1. Download the image (using a random seed to get variations if source supports it,
    # or just the base URL for stability in this demo)
    try:
        if not os.path.exists(file_path):
            # Adding a random query param to ensure we get different bytes/cache bust if needed
            # For this simple demo, we reuse the 3 URLs above to ensure they work.
            r = requests.get(urls[primary_cat], timeout=10)
            with open(file_path, 'wb') as f:
                f.write(r.content)
    except Exception as e:
        print(f"Failed to download {filename}: {e}")
        continue

    # 2. Generate Multi-Labels (One-Hot)
    # Logic: The primary category is always 1. Others have a 10% chance of being 1.
    labels = {}
    for cat in categories:
        if cat == primary_cat:
            labels[cat] = 1
        else:
            labels[cat] = 1 if random.random() > 0.9 else 0

    # Add to records
    record = {"Filename": filename}
    record.update(labels)
    data_records.append(record)

    print(f"Downloaded {filename} -> {labels}")

# --- SAVE CSV ---
df = pd.DataFrame(data_records)
# Reorder columns to ensure Filename is first
cols = ["Filename"] + categories
df = df[cols]
df.to_csv(csv_path, index=False)

print("-" * 30)
print("Download Complete!")
print(f"Images: {images_dir}")
print(f"Labels: {csv_path}")
print("-" * 30)

In [ ]:
train_images_folder = "multi_label_dataset/Images";
train_csv = "multi_label_dataset/train_labels.csv";

In [ ]:
# Create a custom Image Dataset Class

#img_list = list of images
#label_list = list of labels in the same order
#prefix = relative path to images folder

class DatasetMultiClassMultiLabelCSV(Dataset):
    def __init__(self, img_list, label_list, prefix, transform=None):
        self.img_list = img_list;
        self.label_list = label_list;
        self.transform = transform;
        self.prefix = prefix;
        self.classes = self.get_classes();
        self.class_to_idx = {};
        for i in range(len(self.classes)):
            self.class_to_idx[self.classes[i]] = i;

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, index):
        image_name = self.prefix + "/" + self.img_list[index];
        image = Image.open(image_name).convert('RGB');
        label = self.get_one_hot_label(self.label_list[index]);
        if self.transform is not None:
            image = self.transform(image);
        return image, label

    def get_classes(self):
        classes = [];
        for i in range(len(self.label_list)):
            tmp = self.label_list[i].split(",");
            for j in range(len(tmp)):
                if tmp[j] not in classes:
                    classes.append(tmp[j]);
        return sorted(classes)

    def get_one_hot_label(self, label_list):
        label = [];
        for i in range(len(self.classes)):
            if(self.classes[i] in label_list):
                label.append(1);
            else:
                label.append(0);
        return label



In [ ]:
# Read the csv

import pandas as pd
df = pd.read_csv(train_csv);

img_list = [];
label_list = [];

for i in range(len(df)):
    img_list.append(df["Filename"][i]);
    # Construct the multi-label string from one-hot encoded columns
    current_labels = []
    for category in categories: # 'categories' is defined in cell 'tYCtIDcSooov'
        if df[category][i] == 1:
            current_labels.append(category)
    label_list.append(",".join(current_labels));

In [ ]:
label_list[0]

In [ ]:
#transformations

#setting ToTensor() commented for better visualization
data_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        #transforms.ToTensor(),
        #transforms.Normalize(mean=[0.485, 0.456, 0.406],
        #                     std=[0.229, 0.224, 0.225])
    ])

#Using Image Folder Function
train_data = DatasetMultiClassMultiLabelCSV(img_list, label_list, train_images_folder,
                                           transform=data_transform)

In [ ]:
# Getting class list
class_list =  train_data.classes
print(class_list)

In [ ]:
# Image class names to IDs
class_to_id = train_data.class_to_idx
print(class_to_id)

In [ ]:
# Getting list of transforms applied
transforms_list = train_data.transform
print(transforms_list)

In [ ]:
for image, label in train_data:
    print(image.size, label);
    plt.imshow(image)

    break;

In [ ]:
#transformations
#uncommenting ToTensor()
data_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

In [ ]:
#Using Image Folder Function
train_data = DatasetMultiClassCSV(img_list, label_list, train_images_folder,
                                           transform=data_transform)

In [ ]:
#Using DataLoader
train_loader = torch.utils.data.DataLoader(train_data,
                                             batch_size=1, shuffle=True,
                                             num_workers=4)

In [ ]:
#batch size = 2
for data, labels in train_loader:
    print(data.shape, labels.shape);

    break;